#### import library

In [3]:
# import library

import pygame as pyg
import numpy as np
import pandas as pd
import math
import time

from pygame.locals import *
from pygame.rect import *

pygame 2.0.1 (SDL 2.0.14, Python 3.8.6)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [4]:
# Set the color : pyg.Color(Red value, Green value, Blue value)

RED = pyg.Color(255, 0, 0)
GRAY = pyg.Color(128, 128, 128)
GREEN = pyg.Color(0,255,0)
WHITE = pyg.Color(255,255,255)
BLACK = pyg.Color(0,0,0)

#### Define Environment Class

In [3]:
class Environment:
    
    def __init__(self, subj_name):
        self.name = subj_name # 피험자 이름 설정
        self.running = True # Trigger // False시 while문 종료
        self.clock = pyg.time.Clock() # 타임 클럭 설정 // Hz라고 생각하면 편함.
        
        self.count = 0 # 1/120초마다 1씩 증가
        self.duration = 360 # self.duration / 120 = duration이 될 것. (e.g. 360/120 = 3 -> duration은 3초)
        
        self.Screen = Screen() # Screen 클래스를 self.Screen으로 정의
        self.Target = Target() # Target 클래스를 self.Target으로 정의
        self.Joystick = Joystick() # Joystick 클래스를 self.Joystick으로 정의
        
        
    def run(self):
        pyg.init() # pygame start
        self.Screen.open_console() # set the console window
        self.Target.initialize_rect() # pyg.Rect 설정 # Target과 cursor 표시를 위한 기초 작업
        
        while self.running:
            
            # 콘솔 창의 닫기 버튼을 누르면 while문 종료시킨다.
            for event in pyg.event.get():
                if event.type == QUIT:
                    self.running = False
            
            pyg.event.pump() # 이게 있어야 조이스틱이 x, y값을 갱신
            
            self.clock.tick(120) # Hz를 60으로 설정
            self.Screen.over_write() # 콘솔 back ground 색 설정.
            self.Target.move() #Target 움직이게 하는 function
            
            x, y = self.Joystick.reflexion() # cursor를 갱신할 x, y 좌표를 어떻게 입력할 것인가에 대한 function
            
            self.Target.cursor(self.Screen.screen, x, y) # 바로 위 코드에서 받은 x,y를 기준으로 cursor 위치 갱신 
            
            target_pos = list(self.Target.current_pos) # Target의 x, y 좌표륾 받음.
            
            
            if x >=  target_pos[0] and x <=  target_pos[0] + 100 and y >=  target_pos[1] and y <=  target_pos[1] + 100:
                
                self.Target.hit(self.Screen.screen)
                
            else:
                
                if (self.count // self.duration) % 3 == 0: # Hz가 60이므로 self.count가 180이면 3초가 지났음을 의미. 그리고 3초가 짝수의 개수라면 denovo learning을 , 홀수라면 adaptation learning을.
                    self.Target.mode_denovo(self.Screen.screen)
                    
                elif (self.count // self.duration) % 3 == 1:
                    self.Target.mode_adap(self.Screen.screen)
                    
                else:
                    self.Target.no_perturbation(self.Screen.screen)
                    
            self.Screen.flip()
            self.count +=1
            
        pyg.quit()

#### Define Screen Class

In [6]:
class Screen:

    '''
    Class
    
        Name : Screen
        Function : 1) Set the console window 
                   2) Flip the console window
    
    Method
    
        1) open_console
        
         - This function will make a console window 
           which width and heights can be set by a experimenter. 
        
         - Parameters : width, height
         - default : width = 1440, height = 850
        
        2) over_write
        
        - This function will set the color of the display.
        
        - Parameters : color
        - default : color = BLACK 
        
        3) flip
        
        - This function will update the display.
        
        - Parameters : None
    
    '''
    
    def __init__(self):
        pass
    
    def open_console(self, width = 1440, height = 850):

        self.width = width
        self.height = height
        self.size = self.width, self.height
        self.screen = pyg.display.set_mode(self.size)
        
        # 스크린 배경 색
    def over_write(self, color = BLACK):
        self.screen.fill(color)
    
        # 화면 전환
    def flip(self):
        pyg.display.flip()    

#### Define Target Class

In [5]:
class Target:
    
    '''
    Class
    
        Name : Target
        Function : 1) Display a target & cursor
                   2) Set the modes of target & cursor 
    
    Method
    
        1) move
        
        - This move method is responsible to move target.
          In the pygame library, you can set the values of aixs x and y.
          
        - parameters : speed ( type : list)
        - default : speed = [1, 1]
        
        2) cursor
        
        - This cursor method will make an circle for cursor.
          Cursor will move based on x and y
        
        - parameters : x, y
        
        3) initialize_rect
        
        - This method is responsible to target's size.
        
        - Rect(x, y, width, height)
        
        
    '''
    
    def __init__(self):
        pass
        
        
    def move(self, speed = [1, 1]): 
        
        self.speed = speed
        self.rect.move_ip(self.speed)
        self.current_pos = self.rect.copy()
        
        
        if self.rect.left < 0:
            self.speed[0] *= -1
        if self.rect.right > 1440:
            self.speed[0] *= -1
        if self.rect.top < 0:
            self.speed[1] *= -1
        if self.rect.bottom > 850:
            self.speed[1] *= -1
        
    
    def cursor(self,screen, x , y):
        
        pyg.draw.ellipse(screen, GREEN, Rect(x, y, 20, 20), width = 3)
        
    def initialize_rect(self):
        self.rect = Rect(500, 500, 100, 100) # rect의 ( x좌표 , y좌표, 폭, 높이 )
        
    def hit(self,screen):
        pyg.draw.ellipse(screen, WHITE, self.rect, width = 8)
    
    def mode_denovo(self,screen):
        pyg.draw.ellipse(screen, RED, self.rect, width = 8)
        
    def mode_adap(self,screen):
        pyg.draw.ellipse(screen, GREEN, self.rect, width = 8)
        
    def no_perturbation(self,screen):
        pyg.draw.ellipse(screen, GRAY, self.rect, width = 8)

In [6]:
class Joystick:
    
    def __init__(self):
        pyg.joystick.init()
        self.joystick = pyg.joystick.Joystick(0)
        self.joystick.init()
        print(self.joystick.get_init())
        self.curr_x = 700
        self.curr_y = 500
        
    def get_pos(self):
        pi_val = 3.14159
        prev_x = self.joystick.get_axis(0) * 2
        prev_y = self.joystick.get_axis(1) * 2
        
        
        prev_x_rot = np.cos(pi_val/3)*prev_x -np.sin(pi_val/3)*prev_y
        prev_y_rot = np.sin(pi_val/3)*prev_x + np.cos(pi_val/3)*prev_y
        
        
        self.curr_x += prev_x_rot
        self.curr_y += prev_y_rot
        
        return self.curr_x, self.curr_y
    
    def reflexion(self):
        
        prev_x = self.joystick.get_axis(0) * 2
        prev_y = self.joystick.get_axis(1) * 2
        
        phi = np.arctan2(prev_x,prev_y)
        theta = 30
        
        prev_x_reverse = np.cos(2 * phi + theta) + np.sin(2 * phi + theta)
        prev_y_reverse = np.sin(2 * phi + theta) - np.cos(2 * phi + theta)
        
        self.curr_x += prev_x_reverse
        self.curr_y += prev_y_reverse
        
        return self.curr_x, self.curr_y
    
    
    def __str__(self):
        
        return 'Axis x : {:.3f}, Axis y : {:.3f}'.format(self.joystick.get_axis(0),self.joystick.get_axis(1))

In [7]:
env = Environment('lim')

True


In [9]:
env.run()